# Discovery y perfilado de datos

Objetivo del notebook:

- Cargar los 18 archivos CSV de los dominios University, Billing y CRM
- Analizar estructura, volumen y tipos de datos
- Detectar nulos y duplicados
- Validar PKs y FKs
- Analizar variables categóricas y temporales
- Evaluar reglas preliminares de calidad y negocio
- Documentar hallazgos para la construcción de las capas Bronze y Silver

Las transformaciones realizadas en este notebook son únicamente exploratorias.
Los datos crudos no serán modificados.

In [ ]:
import pandas as pd
from  pathlib import Path 

#from google.colab import drive
#drive.mount('/content/drive')

dir_actual = Path.cwd()

# Si se ejecuta desde /notebooks, sube a la raíz del proyecto
if dir_actual.name == "notebooks":
    RUTA_PROYECTO = dir_actual.parent
else:
    RUTA_PROYECTO = dir_actual

RUTA_PATH = (RUTA_PROYECTO / "data" / "raw").resolve()

UNIVERSITY_PATH = RUTA_PATH / "university"
BILLING_PATH = RUTA_PATH / "billing"
CRM_PATH = RUTA_PATH / "crm"

print("Directorio actual:", dir_actual)
print("Raíz del proyecto:", RUTA_PROYECTO)
print("Ruta de datos:", RUTA_PATH)

print("Ruta University:", UNIVERSITY_PATH)
print("Ruta Billing:", BILLING_PATH)
print("Ruta CRM:", CRM_PATH)

rutas_dominios = [
    UNIVERSITY_PATH,
    BILLING_PATH,
    CRM_PATH
]

for ruta_dominio in rutas_dominios:

    if ruta_dominio.exists():
        print("Ruta encontrada:", ruta_dominio)
    else:
        print("Ruta no encontrada:", ruta_dominio)

# UNIVERSITY
df_students = pd.read_csv(UNIVERSITY_PATH / "students.csv")
df_professors = pd.read_csv(UNIVERSITY_PATH / "professors.csv")
df_courses = pd.read_csv(UNIVERSITY_PATH / "courses.csv")
df_semesters = pd.read_csv(UNIVERSITY_PATH / "semesters.csv")
df_enrollments = pd.read_csv(UNIVERSITY_PATH / "enrollments.csv")
df_grades = pd.read_csv(UNIVERSITY_PATH / "grades.csv")

# BILLING
df_customers = pd.read_csv(BILLING_PATH / "customers.csv")
df_products = pd.read_csv(BILLING_PATH / "products.csv")
df_subscriptions = pd.read_csv(BILLING_PATH / "subscriptions.csv")
df_invoices = pd.read_csv(BILLING_PATH / "invoices.csv")
df_invoice_items = pd.read_csv(BILLING_PATH / "invoice_items.csv")
df_payments = pd.read_csv(BILLING_PATH / "payments.csv")

# CRM
df_accounts = pd.read_csv(CRM_PATH / "accounts.csv")
df_contacts = pd.read_csv(CRM_PATH / "contacts.csv")
df_leads = pd.read_csv(CRM_PATH / "leads.csv")
df_opportunities = pd.read_csv(CRM_PATH / "opportunities.csv")
df_opportunity_contacts = pd.read_csv(CRM_PATH / "opportunity_contacts.csv")
df_activities = pd.read_csv(CRM_PATH / "activities.csv")

print("Los 18 archivos fueron cargados correctamente")

Ruta University: /home/florencia/Escritorio/hackatonTigo 15-07/BootCamp V2/notebooks/data/raw/university
Ruta Billing: /home/florencia/Escritorio/hackatonTigo 15-07/BootCamp V2/notebooks/data/raw/billing
Ruta CRM: /home/florencia/Escritorio/hackatonTigo 15-07/BootCamp V2/notebooks/data/raw/crm
Ruta no encontrada: /home/florencia/Escritorio/hackatonTigo 15-07/BootCamp V2/notebooks/data/raw/university
Ruta no encontrada: /home/florencia/Escritorio/hackatonTigo 15-07/BootCamp V2/notebooks/data/raw/billing
Ruta no encontrada: /home/florencia/Escritorio/hackatonTigo 15-07/BootCamp V2/notebooks/data/raw/crm


FileNotFoundError: [Errno 2] No such file or directory: '/home/florencia/Escritorio/hackatonTigo 15-07/BootCamp V2/notebooks/data/raw/university/students.csv'

Se crea un diccionario sencillo para automatizar el perfilado

In [4]:
tablas = {
    "students": df_students,
    "professors": df_professors,
    "courses": df_courses,
    "semesters": df_semesters,
    "enrollments": df_enrollments,
    "grades": df_grades,

    "customers": df_customers,
    "products": df_products,
    "subscriptions": df_subscriptions,
    "invoices": df_invoices,
    "invoice_items": df_invoice_items,
    "payments": df_payments,

    "accounts": df_accounts,
    "contacts": df_contacts,
    "leads": df_leads,
    "opportunities": df_opportunities,
    "opportunity_contacts": df_opportunity_contacts,
    "activities": df_activities
}

print("Cantidad de tablas:", len(tablas))

Cantidad de tablas: 18


In [5]:
resumen_tablas = []

for nombre, df in tablas.items():

    resumen_tablas.append({
        "tabla": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "nulos_totales": df.isnull().sum().sum(),
        "duplicados": df.duplicated().sum()
    })

resumen_tablas = pd.DataFrame(resumen_tablas)

resumen_tablas

,tabla,filas,columnas,nulos_totales,duplicados
0,students,5000,7,0,0
1,professors,200,6,0,0
2,courses,300,6,0,0
3,semesters,8,6,0,0
4,enrollments,25000,6,0,0
5,grades,60000,6,0,0
6,customers,10000,8,5000,0
7,products,200,6,0,0
8,subscriptions,15000,6,0,0
9,invoices,50000,7,0,0


## Perfilado por columnas

Se revisan los tipos de datos detectados por Pandas, la cantidad de valores
nulos y únicos de cada columna.

In [6]:
perfil_columnas = []

for nombre_tabla, df in tablas.items():

    for columna in df.columns:

        perfil_columnas.append({
            "tabla": nombre_tabla,
            "columna": columna,
            "tipo_actual": str(df[columna].dtype),
            "cantidad_nulos": df[columna].isnull().sum(),
            "porcentaje_nulos": round(
                df[columna].isnull().mean() * 100,
                2
            ),
            "valores_unicos": df[columna].nunique()
        })

perfil_columnas = pd.DataFrame(perfil_columnas)

display(perfil_columnas)
print("########################################################################################")
columnas_con_nulos = perfil_columnas[
    perfil_columnas["cantidad_nulos"] > 0
]

display(
    columnas_con_nulos.sort_values(
        "porcentaje_nulos",
        ascending=False
    )
)

,tabla,columna,tipo_actual,cantidad_nulos,porcentaje_nulos,valores_unicos
0,students,student_id,str,0,0.00,5000
1,students,first_name,str,0,0.00,50
2,students,last_name,str,0,0.00,50
3,students,email,str,0,0.00,5000
4,students,birth_date,str,0,0.00,3109
...,...,...,...,...,...,...
109,activities,type,str,0,0.00,5
110,activities,subject,str,0,0.00,20000
111,activities,occurred_at,str,0,0.00,19996
112,activities,contact_id,str,5976,29.88,9085


########################################################################################


,tabla,columna,tipo_actual,cantidad_nulos,porcentaje_nulos,valores_unicos
38,customers,external_ref,str,5000,50.00,5000
113,activities,opportunity_id,str,9985,49.92,2895
112,activities,contact_id,str,5976,29.88,9085


## Validación de llaves primarias

En esta sección se validan las llaves primarias identificadas al modelar el ER

In [7]:
llaves_primarias = {
    # UNIVERSITY
    "students": ["student_id"],
    "professors": ["professor_id"],
    "courses": ["course_id"],
    "semesters": ["semester_id"],
    "enrollments": ["enrollment_id"],
    "grades": ["grade_id"],

    # BILLING
    "customers": ["customer_id"],
    "products": ["product_id"],
    "subscriptions": ["subscription_id"],
    "invoices": ["invoice_id"],
    "invoice_items": ["invoice_item_id"],
    "payments": ["payment_id"],

    # CRM
    "accounts": ["account_id"],
    "contacts": ["contact_id"],
    "leads": ["lead_id"],
    "opportunities": ["opportunity_id"],
    "activities": ["activity_id"],

    # Esta tabla tiene una llave primaria compuesta
    "opportunity_contacts": ["opportunity_id", "contact_id"]
}

opportunity_id + contact_id es diferente porque su llave está formada por dos columnas, esto significa que "**Una oportunidad** puede tener **N contactos** y **un contacto** puede participar en **N oportunidades**"

In [8]:
reporte_pk = []

for nombre_tabla, columnas_pk in llaves_primarias.items():

    df = tablas[nombre_tabla]

    # PKs nulas
    pk_nulas = df[columnas_pk].isnull().any(axis=1).sum()

    # PKs repetidas
    pk_duplicadas = df.duplicated(
        subset=columnas_pk,
        keep=False
    ).sum()

    # Cantidad de combinaciones únicas de la PK
    pk_unicas = df[columnas_pk].drop_duplicates().shape[0]

    if pk_nulas == 0 and pk_duplicadas == 0:
        estado = "OK"
    else:
        estado = "REVISAR"

    reporte_pk.append({
        "tabla": nombre_tabla,
        "llave_primaria": " + ".join(columnas_pk),
        "cantidad_filas": df.shape[0],
        "pk_unicas": pk_unicas,
        "pk_nulas": pk_nulas,
        "filas_con_pk_duplicada": pk_duplicadas,
        "estado": estado
    })

reporte_pk = pd.DataFrame(reporte_pk)

display(reporte_pk)

,tabla,llave_primaria,cantidad_filas,pk_unicas,pk_nulas,filas_con_pk_duplicada,estado
0,students,student_id,5000,5000,0,0,OK
1,professors,professor_id,200,200,0,0,OK
2,courses,course_id,300,300,0,0,OK
3,semesters,semester_id,8,8,0,0,OK
4,enrollments,enrollment_id,25000,25000,0,0,OK
5,grades,grade_id,60000,60000,0,0,OK
6,customers,customer_id,10000,10000,0,0,OK
7,products,product_id,200,200,0,0,OK
8,subscriptions,subscription_id,15000,15000,0,0,OK
9,invoices,invoice_id,50000,50000,0,0,OK


Se evaluaron las llaves primarias de las 18 tablas considerando valores nulos y duplicados

Las tablas con estado "OK" cumplen con las condiciones de unicidad y obligatoriedad esperadas para una llave primaria

COmo no existen tablas con estado "REVISAR" no se hará nada

## Validación de llaves foráneas

En esta sección se validan las relaciones identificadas en el modelo ER.

In [9]:
relaciones_fk = [
    # UNIVERSITY
    {
        "tabla_hija": "courses",
        "columna_fk": "professor_id",
        "tabla_padre": "professors",
        "columna_pk": "professor_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "enrollments",
        "columna_fk": "student_id",
        "tabla_padre": "students",
        "columna_pk": "student_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "enrollments",
        "columna_fk": "course_id",
        "tabla_padre": "courses",
        "columna_pk": "course_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "enrollments",
        "columna_fk": "semester_id",
        "tabla_padre": "semesters",
        "columna_pk": "semester_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "grades",
        "columna_fk": "enrollment_id",
        "tabla_padre": "enrollments",
        "columna_pk": "enrollment_id",
        "permite_nulos": False
    },

    # BILLING
    {
        "tabla_hija": "subscriptions",
        "columna_fk": "customer_id",
        "tabla_padre": "customers",
        "columna_pk": "customer_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "subscriptions",
        "columna_fk": "product_id",
        "tabla_padre": "products",
        "columna_pk": "product_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "invoices",
        "columna_fk": "customer_id",
        "tabla_padre": "customers",
        "columna_pk": "customer_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "invoice_items",
        "columna_fk": "invoice_id",
        "tabla_padre": "invoices",
        "columna_pk": "invoice_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "invoice_items",
        "columna_fk": "product_id",
        "tabla_padre": "products",
        "columna_pk": "product_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "payments",
        "columna_fk": "invoice_id",
        "tabla_padre": "invoices",
        "columna_pk": "invoice_id",
        "permite_nulos": False
    },

    # CRM
    {
        "tabla_hija": "contacts",
        "columna_fk": "account_id",
        "tabla_padre": "accounts",
        "columna_pk": "account_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "opportunities",
        "columna_fk": "account_id",
        "tabla_padre": "accounts",
        "columna_pk": "account_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "opportunity_contacts",
        "columna_fk": "opportunity_id",
        "tabla_padre": "opportunities",
        "columna_pk": "opportunity_id",
        "permite_nulos": False
    },
    {
        "tabla_hija": "opportunity_contacts",
        "columna_fk": "contact_id",
        "tabla_padre": "contacts",
        "columna_pk": "contact_id",
        "permite_nulos": False
    },

    # En activities las dos relaciones son opcionales
    {
        "tabla_hija": "activities",
        "columna_fk": "contact_id",
        "tabla_padre": "contacts",
        "columna_pk": "contact_id",
        "permite_nulos": True
    },
    {
        "tabla_hija": "activities",
        "columna_fk": "opportunity_id",
        "tabla_padre": "opportunities",
        "columna_pk": "opportunity_id",
        "permite_nulos": True
    }
]

Las relaciones de "activities" permiten nulos porque una actividad puede estar vinculada a:


*   A un contacto
*   A una oportunidad
*   A ambos
*   A ninguno





In [10]:
reporte_fk = []

for relacion in relaciones_fk:

    tabla_hija = relacion["tabla_hija"]
    columna_fk = relacion["columna_fk"]
    tabla_padre = relacion["tabla_padre"]
    columna_pk = relacion["columna_pk"]
    permite_nulos = relacion["permite_nulos"]

    df_hija = tablas[tabla_hija]
    df_padre = tablas[tabla_padre]

    # Valores FK
    valores_fk = df_hija[columna_fk]

    # Valores válidos existentes en la tabla padre
    valores_pk = df_padre[columna_pk].dropna().unique()

    # Cantidad de nulos
    cantidad_nulos = valores_fk.isnull().sum()

    # Una llave es huérfana cuando no es nula
    # y tampoco existe en la tabla padre
    mascara_huerfanos = (
        valores_fk.notnull() &
        ~valores_fk.isin(valores_pk)
    )

    cantidad_huerfanos = mascara_huerfanos.sum()

    cantidad_no_nulos = valores_fk.notnull().sum()

    if cantidad_no_nulos > 0:
        porcentaje_huerfanos = round(
            cantidad_huerfanos / cantidad_no_nulos * 100,
            2
        )
    else:
        porcentaje_huerfanos = 0

    # Determinar el estado de la relación
    if cantidad_huerfanos > 0:
        estado = "REVISAR"
    elif cantidad_nulos > 0 and permite_nulos == False:
        estado = "REVISAR"
    else:
        estado = "OK"

    reporte_fk.append({
        "tabla_hija": tabla_hija,
        "columna_fk": columna_fk,
        "tabla_padre": tabla_padre,
        "columna_pk": columna_pk,
        "permite_nulos": permite_nulos,
        "cantidad_filas": df_hija.shape[0],
        "valores_nulos": cantidad_nulos,
        "valores_huerfanos": cantidad_huerfanos,
        "porcentaje_huerfanos": porcentaje_huerfanos,
        "estado": estado
    })

reporte_fk = pd.DataFrame(reporte_fk)

display(reporte_fk)

,tabla_hija,columna_fk,tabla_padre,columna_pk,permite_nulos,cantidad_filas,valores_nulos,valores_huerfanos,porcentaje_huerfanos,estado
0,courses,professor_id,professors,professor_id,False,300,0,0,0.0,OK
1,enrollments,student_id,students,student_id,False,25000,0,0,0.0,OK
2,enrollments,course_id,courses,course_id,False,25000,0,0,0.0,OK
3,enrollments,semester_id,semesters,semester_id,False,25000,0,0,0.0,OK
4,grades,enrollment_id,enrollments,enrollment_id,False,60000,0,0,0.0,OK
5,subscriptions,customer_id,customers,customer_id,False,15000,0,0,0.0,OK
6,subscriptions,product_id,products,product_id,False,15000,0,0,0.0,OK
7,invoices,customer_id,customers,customer_id,False,50000,0,0,0.0,OK
8,invoice_items,invoice_id,invoices,invoice_id,False,150000,0,0,0.0,OK
9,invoice_items,product_id,products,product_id,False,150000,0,0,0.0,OK


### Validación de la relación lógica entre Billing y University

Se analiza la relación entre "customers.external_ref" y
"students.student_id"

Esta relación no corresponde a una llave foránea física del sistema origen, pero según el análisis puede funcionar como una llave de integración entre ambos dominios.

In [11]:
external_ref_no_nulos = df_customers[
    "external_ref"
].dropna()

student_ids = df_students[
    "student_id"
].dropna()

external_ref_en_students = external_ref_no_nulos.isin(
    student_ids
)

cantidad_external_ref = external_ref_no_nulos.shape[0]

cantidad_coincidencias = external_ref_en_students.sum()

cantidad_no_coincidencias = (
    ~external_ref_en_students
).sum()

porcentaje_coincidencias = round(
    cantidad_coincidencias /
    cantidad_external_ref * 100,
    2
)

print("Cantidad de external_ref no nulos:", cantidad_external_ref)
print("Coincidencias con student_id:", cantidad_coincidencias)
print("Sin coincidencia:", cantidad_no_coincidencias)
print("Porcentaje de coincidencia:", porcentaje_coincidencias, "%")

Cantidad de external_ref no nulos: 5000
Coincidencias con student_id: 5000
Sin coincidencia: 0
Porcentaje de coincidencia: 100.0 %


In [12]:
print(
    "External_ref duplicados:",
    external_ref_no_nulos.duplicated().sum()
)

External_ref duplicados: 0


Se validaron las relaciones entre las tablas de los dominios University, Billing y CRM.

Las relaciones con estado "OK" no presentan llaves foráneas huérfanas y cumplen con la obligatoriedad o nulabilidad definida en el modelo ER.

Posteriormente, se validó la relación lógica entre
"customers.external_ref" y "students.student_id", con el objetivo de evaluar su uso como llave de integración entre Billing y University.

## Análisis de valores categóricos

En esta sección se analizan las columnas que representan categorías o clasificaciones del negocio.

El objetivo es identificar:

- Cantidad de categorías diferentes.
- Frecuencia de cada categoría.
- Porcentaje que representa cada valor.
- Valores nulos.
- Posibles diferencias de escritura o categorías inesperadas.

In [13]:
columnas_categoricas = {
    # UNIVERSITY
    "students": ["country"],
    "professors": ["department"],
    "courses": ["department"],
    "semesters": ["half"],
    "enrollments": ["status"],
    "grades": ["assessment"],

    # BILLING
    "customers": ["country", "segment"],
    "products": ["category", "active"],
    "subscriptions": ["status"],
    "invoices": ["status", "currency"],
    "payments": ["method"],

    # CRM
    "accounts": ["industry", "country"],
    "contacts": ["title"],
    "leads": ["source", "status"],
    "opportunities": ["stage"],
    "opportunity_contacts": ["role"],
    "activities": ["type"]
}

In [14]:
resumen_categoricas = []

for nombre_tabla, columnas in columnas_categoricas.items():

    df = tablas[nombre_tabla]

    for columna in columnas:

        valores_ejemplo = (
            df[columna]
            .dropna()
            .astype(str)
            .unique()[:10]
        )

        resumen_categoricas.append({
            "tabla": nombre_tabla,
            "columna": columna,
            "cantidad_filas": df.shape[0],
            "valores_unicos": df[columna].nunique(),
            "valores_nulos": df[columna].isnull().sum(),
            "ejemplos": ", ".join(valores_ejemplo)
        })

resumen_categoricas = pd.DataFrame(resumen_categoricas)

display(resumen_categoricas)

,tabla,columna,cantidad_filas,valores_unicos,valores_nulos,ejemplos
0,students,country,5000,8,0,"US, CL, PE, AR, ES, BR, CO, MX"
1,professors,department,200,8,0,"cs, math, physics, chemistry, economics, histo..."
2,courses,department,300,8,0,"cs, math, literature, physics, biology, histor..."
3,semesters,half,8,2,0,"1, 2"
4,enrollments,status,25000,4,0,"completed, dropped, active, failed"
5,grades,assessment,60000,5,0,"project, homework, quiz, final, midterm"
6,customers,country,10000,8,0,"CL, MX, AR, PE, ES, US, BR, CO"
7,customers,segment,10000,3,0,"smb, retail, enterprise"
8,products,category,200,4,0,"basic, premium, standard, enterprise"
9,products,active,200,2,0,"True, False"


In [15]:
distribucion_categoricas = []

for nombre_tabla, columnas in columnas_categoricas.items():

    df = tablas[nombre_tabla]

    for columna in columnas:

        conteos = (
            df[columna]
            .fillna("VALOR_NULO")
            .value_counts(dropna=False)
        )

        for valor, cantidad in conteos.items():

            porcentaje = round(
                cantidad / df.shape[0] * 100,
                2
            )

            distribucion_categoricas.append({
                "tabla": nombre_tabla,
                "columna": columna,
                "valor": valor,
                "cantidad": cantidad,
                "porcentaje": porcentaje
            })

distribucion_categoricas = pd.DataFrame(
    distribucion_categoricas
)

display(distribucion_categoricas)

,tabla,columna,valor,cantidad,porcentaje
0,students,country,CL,1980,39.60
1,students,country,AR,523,10.46
2,students,country,PE,513,10.26
3,students,country,MX,483,9.66
4,students,country,BR,427,8.54
...,...,...,...,...,...
117,activities,type,email,6904,34.52
118,activities,type,call,6093,30.46
119,activities,type,meeting,3016,15.08
120,activities,type,note,2028,10.14


In [16]:
def ver_distribucion(nombre_tabla, nombre_columna):

    resultado = distribucion_categoricas[
        (distribucion_categoricas["tabla"] == nombre_tabla) &
        (distribucion_categoricas["columna"] == nombre_columna)
    ]

    return resultado.sort_values(
        "cantidad",
        ascending=False
    )


Ejemplo de analizar una tabla y columna específica

In [17]:
display(
    ver_distribucion(
        "invoices",
        "status"
    )
)

,tabla,columna,valor,cantidad,porcentaje
55,invoices,status,paid,34966,69.93
56,invoices,status,pending,10048,20.10
57,invoices,status,overdue,4986,9.97


# Diferencias en la escritura
Esto sirve para revisar las diferencias sin modificar los datos originales (ejm. En "status" puede estar como: active, Active, ACTIVE, active)

In [18]:
comparacion_categorias = []

for nombre_tabla, columnas in columnas_categoricas.items():

    df = tablas[nombre_tabla]

    for columna in columnas:

        valores_originales = df[columna].nunique(
            dropna=True
        )

        valores_normalizados = (
            df[columna]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
            .nunique()
        )

        diferencia = (
            valores_originales -
            valores_normalizados
        )

        if diferencia > 0:
            estado = "REVISAR"
        else:
            estado = "OK"

        comparacion_categorias.append({
            "tabla": nombre_tabla,
            "columna": columna,
            "valores_originales": valores_originales,
            "valores_normalizados": valores_normalizados,
            "posibles_variantes": diferencia,
            "estado": estado
        })

comparacion_categorias = pd.DataFrame(
    comparacion_categorias
)

display(comparacion_categorias)

,tabla,columna,valores_originales,valores_normalizados,posibles_variantes,estado
0,students,country,8,8,0,OK
1,professors,department,8,8,0,OK
2,courses,department,8,8,0,OK
3,semesters,half,2,2,0,OK
4,enrollments,status,4,4,0,OK
5,grades,assessment,5,5,0,OK
6,customers,country,8,8,0,OK
7,customers,segment,3,3,0,OK
8,products,category,4,4,0,OK
9,products,active,2,2,0,OK


Se analizaron las principales variables categóricas de los dominios
University, Billing y CRM.

Para cada columna se identificaron sus valores únicos, frecuencia,
porcentaje de participación y cantidad de valores nulos.

También se compararon los valores originales con diferentes versiones de escritura en minúsculas y sin espacios adicionales, con el propósito de detectar posibles diferencias de escritura.


## Análisis de fechas

En esta sección se analizan las columnas temporales de los tres dominios.

El objetivo es identificar:

- El tipo de dato detectado originalmente por Pandas.
- Valores nulos.
- Fechas con formatos inválidos.
- Fechas mínimas y máximas.
- Inconsistencias temporales entre columnas relacionadas.

In [19]:
columnas_fechas = {
    # UNIVERSITY
    "semesters": ["start_date", "end_date"],
    "professors": ["hired_at"],
    "students": ["birth_date", "enrolled_at"],
    "enrollments": ["enrolled_at"],
    "grades": ["graded_at"],

    # BILLING
    "customers": ["created_at"],
    "subscriptions": ["start_date", "end_date"],
    "invoices": ["issued_at", "due_at"],
    "payments": ["paid_at"],

    # CRM
    "accounts": ["created_at"],
    "contacts": ["created_at"],
    "leads": ["created_at"],
    "opportunities": ["created_at", "close_date"],
    "activities": ["occurred_at"]
}

In [20]:
reporte_fechas = []

for nombre_tabla, columnas in columnas_fechas.items():

    df = tablas[nombre_tabla]

    for columna in columnas:

        # Serie original
        serie_original = df[columna]

        # Conversión temporal solo para el análisis
        serie_convertida = pd.to_datetime(
            serie_original,
            errors="coerce"
        )

        # Un valor es inválido cuando no es nulo originalmente,
        # pero al convertirlo se transforma en NaT
        mascara_invalidos = (
            serie_original.notnull() &
            serie_convertida.isnull()
        )

        cantidad_nulos = serie_original.isnull().sum()
        cantidad_validos = serie_convertida.notnull().sum()
        cantidad_invalidos = mascara_invalidos.sum()

        if serie_convertida.notnull().sum() > 0:
            fecha_minima = serie_convertida.min()
            fecha_maxima = serie_convertida.max()
        else:
            fecha_minima = None
            fecha_maxima = None

        reporte_fechas.append({
            "tabla": nombre_tabla,
            "columna": columna,
            "tipo_actual": str(serie_original.dtype),
            "cantidad_filas": df.shape[0],
            "valores_nulos": cantidad_nulos,
            "fechas_validas": cantidad_validos,
            "fechas_invalidas": cantidad_invalidos,
            "fecha_minima": fecha_minima,
            "fecha_maxima": fecha_maxima
        })

reporte_fechas = pd.DataFrame(reporte_fechas)

display(reporte_fechas)

,tabla,columna,tipo_actual,cantidad_filas,valores_nulos,fechas_validas,fechas_invalidas,fecha_minima,fecha_maxima
0,semesters,start_date,str,8,0,8,0,2022-03-01 00:00:00,2025-08-01 00:00:00
1,semesters,end_date,str,8,0,8,0,2022-07-15 00:00:00,2025-12-15 00:00:00
2,professors,hired_at,str,200,0,200,0,2005-03-10 00:00:00,2024-12-27 00:00:00
3,students,birth_date,str,5000,0,5000,0,1995-01-01 00:00:00,2007-12-31 00:00:00
4,students,enrolled_at,str,5000,0,5000,0,2018-01-01 00:00:00,2025-09-30 00:00:00
5,enrollments,enrolled_at,str,25000,0,25000,0,2022-01-01 00:00:00,2025-12-31 00:00:00
6,grades,graded_at,str,60000,0,60000,0,2022-02-01 00:00:00,2025-12-31 00:00:00
7,customers,created_at,str,10000,0,10000,0,2018-01-01 02:21:56,2025-12-30 13:31:31
8,subscriptions,start_date,str,15000,0,15000,0,2020-01-01 00:00:00,2025-06-30 00:00:00
9,subscriptions,end_date,str,15000,0,15000,0,2024-01-01 00:00:00,2027-12-31 00:00:00


Se puede ver que el tipo de dato que se maneja es "object" que seria como un string, esto deberia transformarse a date o timestamp (en la capa silver)

# Validaciones de fechas
Esto es para verificar si tienen sentido su inicio y fin

In [21]:
inicio_semestre = pd.to_datetime(
    df_semesters["start_date"],
    errors="coerce"
)

fin_semestre = pd.to_datetime(
    df_semesters["end_date"],
    errors="coerce"
)

semestres_inconsistentes = df_semesters[
    inicio_semestre > fin_semestre
]

print(
    "Semestres con fecha de inicio posterior al final:",
    semestres_inconsistentes.shape[0]
)

display(semestres_inconsistentes)

Semestres con fecha de inicio posterior al final: 0


,semester_id,code,year,half,start_date,end_date


In [22]:
fecha_nacimiento = pd.to_datetime(
    df_students["birth_date"],
    errors="coerce"
)

fecha_inscripcion_estudiante = pd.to_datetime(
    df_students["enrolled_at"],
    errors="coerce"
)

estudiantes_fechas_inconsistentes = df_students[
    fecha_nacimiento >= fecha_inscripcion_estudiante
]

print(
    "Estudiantes con fecha de nacimiento posterior "
    "o igual a la inscripción:",
    estudiantes_fechas_inconsistentes.shape[0]
)

display(estudiantes_fechas_inconsistentes)

Estudiantes con fecha de nacimiento posterior o igual a la inscripción: 0


,student_id,first_name,last_name,email,birth_date,enrolled_at,country


In [23]:
edad_al_inscribirse = (
    fecha_inscripcion_estudiante -
    fecha_nacimiento
).dt.days / 365

df_students_revision_fechas = df_students[
    ["student_id", "birth_date", "enrolled_at"]
].copy()

df_students_revision_fechas[
    "edad_aproximada_inscripcion"
] = edad_al_inscribirse.round(1)

display(
    df_students_revision_fechas[
        "edad_aproximada_inscripcion"
    ].describe()
)

count    5000.000000
mean       20.355300
std         4.356478
min        10.200000
25%        17.200000
50%        20.400000
75%        23.600000
max        30.600000
Name: edad_aproximada_inscripcion, dtype: float64

In [24]:
inicio_suscripcion = pd.to_datetime(
    df_subscriptions["start_date"],
    errors="coerce"
)

fin_suscripcion = pd.to_datetime(
    df_subscriptions["end_date"],
    errors="coerce"
)

suscripciones_inconsistentes = df_subscriptions[
    fin_suscripcion.notnull() &
    (inicio_suscripcion > fin_suscripcion)
]

print(
    "Suscripciones con inicio posterior al final:",
    suscripciones_inconsistentes.shape[0]
)

display(suscripciones_inconsistentes)

Suscripciones con inicio posterior al final: 783


,subscription_id,status,start_date,end_date,customer_id,product_id
28,SUB-0000029,active,2024-12-03,2024-02-19,CUS-0005963,PRD-00004
43,SUB-0000044,paused,2024-07-16,2024-07-06,CUS-0009149,PRD-00096
55,SUB-0000056,active,2024-05-27,2024-05-04,CUS-0009361,PRD-00102
61,SUB-0000062,active,2025-05-20,2024-04-12,CUS-0004678,PRD-00165
69,SUB-0000070,active,2025-04-16,2024-07-09,CUS-0003571,PRD-00118
...,...,...,...,...,...,...
14928,SUB-0014929,active,2025-06-10,2024-05-13,CUS-0006966,PRD-00115
14943,SUB-0014944,active,2025-06-10,2024-02-19,CUS-0001092,PRD-00138
14962,SUB-0014963,active,2024-12-22,2024-09-22,CUS-0000266,PRD-00040
14965,SUB-0014966,active,2024-10-31,2024-02-19,CUS-0008734,PRD-00111


In [25]:
fecha_creacion_oportunidad = pd.to_datetime(
    df_opportunities["created_at"],
    errors="coerce"
)

fecha_cierre_oportunidad = pd.to_datetime(
    df_opportunities["close_date"],
    errors="coerce"
)

oportunidades_fechas_inconsistentes = df_opportunities[
    fecha_cierre_oportunidad.notnull() &
    (
        fecha_creacion_oportunidad >
        fecha_cierre_oportunidad
    )
]

print(
    "Oportunidades creadas después de su fecha de cierre:",
    oportunidades_fechas_inconsistentes.shape[0]
)

display(oportunidades_fechas_inconsistentes)

Oportunidades creadas después de su fecha de cierre: 1029


,opportunity_id,name,stage,amount,close_date,created_at,account_id
1,OPP-0000002,Deal 0000002,negotiation,24990.51,2023-09-28,2025-05-25 03:05:15,ACC-0000582
4,OPP-0000005,Deal 0000005,proposal,27434.07,2023-08-30,2024-11-30 06:25:22,ACC-0003484
8,OPP-0000009,Deal 0000009,negotiation,13695.31,2023-08-29,2024-07-08 09:42:27,ACC-0003275
11,OPP-0000012,Deal 0000012,prospect,24708.46,2023-03-03,2025-03-09 20:57:04,ACC-0003139
12,OPP-0000013,Deal 0000013,proposal,8010.86,2023-06-03,2023-10-06 12:46:38,ACC-0000912
...,...,...,...,...,...,...,...
2990,OPP-0002991,Deal 0002991,proposal,36538.06,2024-08-21,2025-03-17 05:26:03,ACC-0000231
2992,OPP-0002993,Deal 0002993,won,5128.88,2023-01-26,2024-07-05 01:49:52,ACC-0004687
2994,OPP-0002995,Deal 0002995,prospect,24711.42,2023-11-14,2025-05-23 12:23:37,ACC-0000368
2995,OPP-0002996,Deal 0002996,negotiation,5607.60,2023-04-23,2024-08-14 03:33:19,ACC-0002961


In [26]:
reglas_temporales = [
    {
        "regla": "SEM-01",
        "tabla": "semesters",
        "descripcion": "start_date debe ser menor o igual a end_date",
        "registros_invalidos": semestres_inconsistentes.shape[0]
    },
    {
        "regla": "STU-01",
        "tabla": "students",
        "descripcion": "birth_date debe ser anterior a enrolled_at",
        "registros_invalidos": estudiantes_fechas_inconsistentes.shape[0]
    },
    {
        "regla": "SUB-01",
        "tabla": "subscriptions",
        "descripcion": "start_date debe ser menor o igual a end_date",
        "registros_invalidos": suscripciones_inconsistentes.shape[0]
    },
    {
        "regla": "OPP-01",
        "tabla": "opportunities",
        "descripcion": "created_at debe ser menor o igual a close_date",
        "registros_invalidos": oportunidades_fechas_inconsistentes.shape[0]
    }
]

reporte_reglas_temporales = pd.DataFrame(
    reglas_temporales
)

reporte_reglas_temporales["estado"] = (
    reporte_reglas_temporales[
        "registros_invalidos"
    ]
    .apply(
        lambda cantidad: "OK"
        if cantidad == 0
        else "REVISAR"
    )
)

display(reporte_reglas_temporales)

,regla,tabla,descripcion,registros_invalidos,estado
0,SEM-01,semesters,start_date debe ser menor o igual a end_date,0,OK
1,STU-01,students,birth_date debe ser anterior a enrolled_at,0,OK
2,SUB-01,subscriptions,start_date debe ser menor o igual a end_date,783,REVISAR
3,OPP-01,opportunities,created_at debe ser menor o igual a close_date,1029,REVISAR


Se identificaron suscripciones cuya fecha de inicio es posterior a su fecha de finalización. Estos registros fueron marcados para revisión, debido a que contradicen la secuencia temporal esperada de una suscripción.

De  la misma forma para oportunidades cuya fecha de creación es posterior a la fecha de cierre. Este resultado puede representar inconsistencias de calidad o diferencias en el significado de close_date, por ejemplo, una fecha estimada o una fecha cargada incorrectamente.

## Tipos de datos actuales y esperados en Silver

En esta sección se comparan los tipos de datos detectados por Pandas con los tipos esperados para la capa Silver.

En Silver se aplicarán conversiones para obtener datos
consistentes y adecuados para consultas y análisis.

In [27]:
tipos_esperados = {
    # UNIVERSITY
    "students": {
        "student_id": "string",
        "first_name": "string",
        "last_name": "string",
        "email": "string",
        "birth_date": "date",
        "enrolled_at": "timestamp",
        "country": "string"
    },

    "professors": {
        "professor_id": "string",
        "first_name": "string",
        "last_name": "string",
        "email": "string",
        "department": "string",
        "hired_at": "timestamp"
    },

    "courses": {
        "course_id": "string",
        "code": "string",
        "name": "string",
        "credits": "integer",
        "department": "string",
        "professor_id": "string"
    },

    "semesters": {
        "semester_id": "string",
        "code": "string",
        "year": "integer",
        "half": "integer",
        "start_date": "date",
        "end_date": "date"
    },

    "enrollments": {
        "enrollment_id": "string",
        "enrolled_at": "timestamp",
        "status": "string",
        "student_id": "string",
        "course_id": "string",
        "semester_id": "string"
    },

    "grades": {
        "grade_id": "string",
        "assessment": "string",
        "score": "decimal",
        "weight": "decimal",
        "graded_at": "timestamp",
        "enrollment_id": "string"
    },

    # BILLING
    "customers": {
        "customer_id": "string",
        "external_ref": "string",
        "first_name": "string",
        "last_name": "string",
        "email": "string",
        "country": "string",
        "created_at": "timestamp",
        "segment": "string"
    },

    "products": {
        "product_id": "string",
        "sku": "string",
        "name": "string",
        "category": "string",
        "monthly_price": "decimal",
        "active": "boolean"
    },

    "subscriptions": {
        "subscription_id": "string",
        "status": "string",
        "start_date": "date",
        "end_date": "date",
        "customer_id": "string",
        "product_id": "string"
    },

    "invoices": {
        "invoice_id": "string",
        "issued_at": "timestamp",
        "due_at": "timestamp",
        "total": "decimal",
        "status": "string",
        "currency": "string",
        "customer_id": "string"
    },

    "invoice_items": {
        "invoice_item_id": "string",
        "quantity": "integer",
        "unit_price": "decimal",
        "line_total": "decimal",
        "invoice_id": "string",
        "product_id": "string"
    },

    "payments": {
        "payment_id": "string",
        "amount": "decimal",
        "paid_at": "timestamp",
        "method": "string",
        "invoice_id": "string"
    },

    # CRM
    "accounts": {
        "account_id": "string",
        "name": "string",
        "industry": "string",
        "country": "string",
        "annual_revenue": "decimal",
        "employees": "integer",
        "created_at": "timestamp"
    },

    "contacts": {
        "contact_id": "string",
        "first_name": "string",
        "last_name": "string",
        "email": "string",
        "phone": "string",
        "title": "string",
        "created_at": "timestamp",
        "account_id": "string"
    },

    "leads": {
        "lead_id": "string",
        "first_name": "string",
        "last_name": "string",
        "email": "string",
        "source": "string",
        "status": "string",
        "score": "integer",
        "created_at": "timestamp"
    },

    "opportunities": {
        "opportunity_id": "string",
        "name": "string",
        "stage": "string",
        "amount": "decimal",
        "close_date": "date",
        "created_at": "timestamp",
        "account_id": "string"
    },

    "opportunity_contacts": {
        "opportunity_id": "string",
        "contact_id": "string",
        "role": "string"
    },

    "activities": {
        "activity_id": "string",
        "type": "string",
        "subject": "string",
        "occurred_at": "timestamp",
        "contact_id": "string",
        "opportunity_id": "string"
    }
}

In [28]:
tipos_sql = {
    "string": "VARCHAR",
    "integer": "INTEGER",
    "decimal": "NUMERIC",
    "boolean": "BOOLEAN",
    "date": "DATE",
    "timestamp": "TIMESTAMP"
}

In [29]:
def identificar_tipo_actual(serie):

    if pd.api.types.is_bool_dtype(serie):
        return "boolean"

    elif pd.api.types.is_integer_dtype(serie):
        return "integer"

    elif pd.api.types.is_float_dtype(serie):
        return "decimal"

    elif pd.api.types.is_datetime64_any_dtype(serie):
        return "timestamp"

    else:
        return "string"

In [30]:
reporte_tipos = []

for nombre_tabla, df in tablas.items():

    for columna in df.columns:

        tipo_pandas = str(df[columna].dtype)
        tipo_actual = identificar_tipo_actual(df[columna])

        if (
            nombre_tabla in tipos_esperados
            and columna in tipos_esperados[nombre_tabla]
        ):
            tipo_esperado = tipos_esperados[
                nombre_tabla
            ][columna]

            tipo_sql_esperado = tipos_sql[
                tipo_esperado
            ]
        else:
            tipo_esperado = "NO DEFINIDO"
            tipo_sql_esperado = "NO DEFINIDO"

        if tipo_actual == tipo_esperado:
            requiere_conversion = "NO"
        else:
            requiere_conversion = "SÍ"

        reporte_tipos.append({
            "tabla": nombre_tabla,
            "columna": columna,
            "tipo_pandas": tipo_pandas,
            "tipo_actual_logico": tipo_actual,
            "tipo_esperado_silver": tipo_esperado,
            "tipo_sql_silver": tipo_sql_esperado,
            "requiere_conversion": requiere_conversion
        })

reporte_tipos = pd.DataFrame(reporte_tipos)

display(reporte_tipos)

,tabla,columna,tipo_pandas,tipo_actual_logico,tipo_esperado_silver,tipo_sql_silver,requiere_conversion
0,students,student_id,str,string,string,VARCHAR,NO
1,students,first_name,str,string,string,VARCHAR,NO
2,students,last_name,str,string,string,VARCHAR,NO
3,students,email,str,string,string,VARCHAR,NO
4,students,birth_date,str,string,date,DATE,SÍ
...,...,...,...,...,...,...,...
109,activities,type,str,string,string,VARCHAR,NO
110,activities,subject,str,string,string,VARCHAR,NO
111,activities,occurred_at,str,string,timestamp,TIMESTAMP,SÍ
112,activities,contact_id,str,string,string,VARCHAR,NO


In [31]:
columnas_para_convertir = reporte_tipos[
    reporte_tipos["requiere_conversion"] == "SÍ"
]

display(columnas_para_convertir)

,tabla,columna,tipo_pandas,tipo_actual_logico,tipo_esperado_silver,tipo_sql_silver,requiere_conversion
4,students,birth_date,str,string,date,DATE,SÍ
5,students,enrolled_at,str,string,timestamp,TIMESTAMP,SÍ
12,professors,hired_at,str,string,timestamp,TIMESTAMP,SÍ
23,semesters,start_date,str,string,date,DATE,SÍ
24,semesters,end_date,str,string,date,DATE,SÍ
26,enrollments,enrolled_at,str,string,timestamp,TIMESTAMP,SÍ
35,grades,graded_at,str,string,timestamp,TIMESTAMP,SÍ
43,customers,created_at,str,string,timestamp,TIMESTAMP,SÍ
53,subscriptions,start_date,str,string,date,DATE,SÍ
54,subscriptions,end_date,str,string,date,DATE,SÍ


Se compararon los tipos de datos detectados por Pandas con los tipos esperados para la capa Silver.

La mayoría de las columnas de fecha fueron leídas inicialmente como texto,por lo que deberán convertirse a `DATE` o `TIMESTAMP`

## Validación de reglas de calidad y negocio

En esta sección se evalúan reglas relacionadas con la coherencia y validez de los datos.

Las reglas fueron planteadas a partir del significado de las columnas y de las relaciones encontradas durante Discovery.


In [32]:
import numpy as np

resultados_reglas = []
ejemplos_reglas = {}

def registrar_regla(
    codigo,
    tabla,
    descripcion,
    df,
    condicion_invalida,
    severidad="ERROR",
    columnas_ejemplo=None
):
    cantidad_evaluada = df.shape[0]
    cantidad_invalida = condicion_invalida.fillna(False).sum()

    if cantidad_evaluada > 0:
        porcentaje_invalido = round(
            cantidad_invalida / cantidad_evaluada * 100,
            2
        )
    else:
        porcentaje_invalido = 0

    if cantidad_invalida == 0:
        estado = "OK"
    else:
        estado = "REVISAR"

    resultados_reglas.append({
        "codigo": codigo,
        "tabla": tabla,
        "descripcion": descripcion,
        "severidad": severidad,
        "registros_evaluados": cantidad_evaluada,
        "registros_invalidos": cantidad_invalida,
        "porcentaje_invalido": porcentaje_invalido,
        "estado": estado
    })

    if columnas_ejemplo is None:
        columnas_ejemplo = df.columns.tolist()

    ejemplos_reglas[codigo] = df.loc[
        condicion_invalida.fillna(False),
        columnas_ejemplo
    ].head(20)

# Reglas de University

Notas entre 0 y 100

In [33]:
registrar_regla(
    codigo="UNI-01",
    tabla="grades",
    descripcion="La calificación score debe estar entre 0 y 100",
    df=df_grades,
    condicion_invalida=~df_grades["score"].between(
        0,
        100,
        inclusive="both"
    ),
    columnas_ejemplo=[
        "grade_id",
        "enrollment_id",
        "assessment",
        "score"
    ]
)

Peso de evaluación entre 0 y 1

El peso representa una proporción:
0.20 = 20%
0.50 = 50%
1.00 = 100%

In [34]:
registrar_regla(
    codigo="UNI-02",
    tabla="grades",
    descripcion="El peso weight debe ser mayor que 0 y menor o igual que 1",
    df=df_grades,
    condicion_invalida=(
        (df_grades["weight"] <= 0) |
        (df_grades["weight"] > 1)
    ),
    columnas_ejemplo=[
        "grade_id",
        "enrollment_id",
        "assessment",
        "weight"
    ]
)

Inscripciones repetidas
Un estudiante no debería aparecer inscrito 2 veces en el mismo curso del semestre

In [35]:
inscripciones_repetidas = df_enrollments.duplicated(
    subset=[
        "student_id",
        "course_id",
        "semester_id"
    ],
    keep=False
)

registrar_regla(
    codigo="UNI-03",
    tabla="enrollments",
    descripcion=(
        "La combinación student_id, course_id y semester_id "
        "no debería repetirse"
    ),
    df=df_enrollments,
    condicion_invalida=inscripciones_repetidas,
    severidad="WARNING",
    columnas_ejemplo=[
        "enrollment_id",
        "student_id",
        "course_id",
        "semester_id",
        "status",
        "enrolled_at"
    ]
)

Suma de pesos por inscripción 
Para cada enrollment_id, los pesos de sus evaluaciones normalmente deberían sumar 1

In [36]:
suma_pesos = (
    df_grades
    .groupby("enrollment_id", as_index=False)["weight"]
    .sum()
    .rename(columns={"weight": "suma_pesos"})
)

suma_pesos["diferencia_con_1"] = (
    suma_pesos["suma_pesos"] - 1
).abs()

suma_pesos["peso_inconsistente"] = ~np.isclose(
    suma_pesos["suma_pesos"],
    1,
    atol=0.01,
    rtol=0
)

registrar_regla(
    codigo="UNI-04",
    tabla="grades agrupado por enrollment_id",
    descripcion="La suma de weight por inscripción debería ser igual a 1",
    df=suma_pesos,
    condicion_invalida=suma_pesos["peso_inconsistente"],
    severidad="WARNING",
    columnas_ejemplo=[
        "enrollment_id",
        "suma_pesos",
        "diferencia_con_1"
    ]
)

# Reglas de Billing

Cantidad positiva en las líneas de factura

In [37]:
registrar_regla(
    codigo="BIL-01",
    tabla="invoice_items",
    descripcion="La cantidad quantity debe ser mayor que cero",
    df=df_invoice_items,
    condicion_invalida=df_invoice_items["quantity"] <= 0,
    columnas_ejemplo=[
        "invoice_item_id",
        "invoice_id",
        "product_id",
        "quantity"
    ]
)

Precio unitario no negativo

In [38]:
registrar_regla(
    codigo="BIL-02",
    tabla="invoice_items",
    descripcion="El precio unit_price no debe ser negativo",
    df=df_invoice_items,
    condicion_invalida=df_invoice_items["unit_price"] < 0,
    columnas_ejemplo=[
        "invoice_item_id",
        "product_id",
        "unit_price"
    ]
)

Validar total de cada línea
line_total = quantity x unit_price
Se utiliza una tolerancia de 0.01 por posibles redondeos

In [39]:
total_linea_calculado = (
    df_invoice_items["quantity"] *
    df_invoice_items["unit_price"]
)

lineas_inconsistentes = ~np.isclose(
    df_invoice_items["line_total"],
    total_linea_calculado,
    atol=0.01,
    rtol=0
)

df_revision_lineas = df_invoice_items.copy()
df_revision_lineas["total_calculado"] = total_linea_calculado
df_revision_lineas["diferencia"] = (
    df_revision_lineas["line_total"] -
    df_revision_lineas["total_calculado"]
).round(2)

registrar_regla(
    codigo="BIL-03",
    tabla="invoice_items",
    descripcion=(
        "line_total debe coincidir con quantity multiplicado "
        "por unit_price"
    ),
    df=df_revision_lineas,
    condicion_invalida=pd.Series(
        lineas_inconsistentes,
        index=df_revision_lineas.index
    ),
    columnas_ejemplo=[
        "invoice_item_id",
        "quantity",
        "unit_price",
        "line_total",
        "total_calculado",
        "diferencia"
    ]
)

Precio mendual de productos

In [40]:
registrar_regla(
    codigo="BIL-04",
    tabla="products",
    descripcion="El precio mensual no debe ser negativo",
    df=df_products,
    condicion_invalida=df_products["monthly_price"] < 0,
    columnas_ejemplo=[
        "product_id",
        "name",
        "monthly_price",
        "active"
    ]
)

Pagos mayores que cero

In [41]:
registrar_regla(
    codigo="BIL-05",
    tabla="payments",
    descripcion="El monto del pago debe ser mayor que cero",
    df=df_payments,
    condicion_invalida=df_payments["amount"] <= 0,
    columnas_ejemplo=[
        "payment_id",
        "invoice_id",
        "amount",
        "paid_at",
        "method"
    ]
)

Conciliación entra facturas y líneas 
Se realiza la comparación de invoice.total con suma de invoice_items.line_total

Primero se calcula el total de las líneas de factura 

In [48]:
totales_por_factura = (
    df_invoice_items
    .groupby("invoice_id", as_index=False)["line_total"]
    .sum()
    .rename(
        columns={
            "line_total": "total_calculado_lineas"
        }
    )
)

Unimos resultados con invoices:

In [49]:
conciliacion_facturas = df_invoices[
    [
        "invoice_id",
        "customer_id",
        "issued_at",
        "status",
        "currency",
        "total"
    ]
].merge(
    totales_por_factura,
    on="invoice_id",
    how="left"
)

In [50]:
conciliacion_facturas["sin_lineas"] = (
    conciliacion_facturas["total_calculado_lineas"].isnull()
)

conciliacion_facturas["diferencia"] = (
    conciliacion_facturas["total"] -
    conciliacion_facturas["total_calculado_lineas"]
).round(2)

conciliacion_facturas["total_coincide"] = np.isclose(
    conciliacion_facturas["total"],
    conciliacion_facturas["total_calculado_lineas"],
    atol=0.01,
    rtol=0,
    equal_nan=False
)

Facturas sin líneas

In [51]:
registrar_regla(
    codigo="BIL-06",
    tabla="invoices + invoice_items",
    descripcion="Toda factura debería tener al menos una línea asociada",
    df=conciliacion_facturas,
    condicion_invalida=conciliacion_facturas["sin_lineas"],
    severidad="WARNING",
    columnas_ejemplo=[
        "invoice_id",
        "customer_id",
        "total",
        "status",
        "sin_lineas"
    ]
)

Diferencia entre total informado y calculado

In [52]:
registrar_regla(
    codigo="BIL-07",
    tabla="invoices + invoice_items",
    descripcion=(
        "El total de la factura debería coincidir con la suma "
        "de line_total"
    ),
    df=conciliacion_facturas,
    condicion_invalida=(
        ~conciliacion_facturas["sin_lineas"] &
        ~conciliacion_facturas["total_coincide"]
    ),
    severidad="WARNING",
    columnas_ejemplo=[
        "invoice_id",
        "total",
        "total_calculado_lineas",
        "diferencia",
        "currency",
        "status"
    ]
)

# Reglas de CRM

Ingresos anuales no negativos

In [42]:
registrar_regla(
    codigo="CRM-01",
    tabla="accounts",
    descripcion="annual_revenue no debe ser negativo",
    df=df_accounts,
    condicion_invalida=df_accounts["annual_revenue"] < 0,
    columnas_ejemplo=[
        "account_id",
        "name",
        "annual_revenue"
    ]
)

La cantidad de empleados debe ser no negativa

In [43]:
registrar_regla(
    codigo="CRM-02",
    tabla="accounts",
    descripcion="employees no debe ser negativo",
    df=df_accounts,
    condicion_invalida=df_accounts["employees"] < 0,
    columnas_ejemplo=[
        "account_id",
        "name",
        "employees"
    ]
)

Score de leads

In [44]:
registrar_regla(
    codigo="CRM-03",
    tabla="leads",
    descripcion="El score del lead debe estar entre 0 y 100",
    df=df_leads,
    condicion_invalida=~df_leads["score"].between(
        0,
        100,
        inclusive="both"
    ),
    columnas_ejemplo=[
        "lead_id",
        "email",
        "status",
        "score"
    ]
)

Monto de oportunidades

In [45]:
registrar_regla(
    codigo="CRM-04",
    tabla="opportunities",
    descripcion="El amount de una oportunidad no debe ser negativo",
    df=df_opportunities,
    condicion_invalida=df_opportunities["amount"] < 0,
    columnas_ejemplo=[
        "opportunity_id",
        "name",
        "stage",
        "amount"
    ]
)

In [53]:
reporte_reglas = pd.DataFrame(resultados_reglas)

reporte_reglas_revisar = reporte_reglas[
    reporte_reglas["estado"] == "REVISAR"
]

display(reporte_reglas_revisar)

,codigo,tabla,descripcion,severidad,registros_evaluados,registros_invalidos,porcentaje_invalido,estado
2,UNI-03,enrollments,"La combinación student_id, course_id y semeste...",WARNING,25000,46,0.18,REVISAR
3,UNI-04,grades agrupado por enrollment_id,La suma de weight por inscripción debería ser ...,WARNING,22786,22645,99.38,REVISAR
14,BIL-06,invoices + invoice_items,Toda factura debería tener al menos una línea ...,WARNING,50000,2502,5.00,REVISAR
15,BIL-07,invoices + invoice_items,El total de la factura debería coincidir con l...,WARNING,50000,47497,94.99,REVISAR


Las reglas con estado "REVISAR" requieren un análisis adicional antes de definir su tratamiento en Silver.